In [ ]:
# Install all required libraries
!pip install -q pymupdf transformers sentencepiece rouge-score tiktoken accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 65.0 MB/s eta 0:00:00


In [ ]:
!pip install -q sentence-transformers

In [ ]:
import fitz
import textwrap
import tiktoken
import torch
from rouge_score import rouge_scorer
from google.colab import files

print("Libraries imported successfully")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

Libraries imported successfully
GPU available: True
Device: Tesla T4


In [ ]:
USE_SAMPLE = False   # Set to False to upload your own PDF

if USE_SAMPLE:
    !wget -q -O sample.pdf https://arxiv.org/pdf/1706.03762  # Attention Is All You Need
    PDF_PATH = "sample.pdf"
    print("Downloaded sample PDF: Attention Is All You Need")
else:
    uploaded = files.upload()
    PDF_PATH = list(uploaded.keys())[0]
    print(f"Uploaded: {PDF_PATH}")

Saving Test PDF (Hard).pdf to Test PDF (Hard).pdf
Uploaded: Test PDF (Hard).pdf


---
# Block 2: PDF Text Extraction
Use PyMuPDF (`fitz`) to extract page-level text, then clean and normalize it.

Discussion: PDFs store content by rendering order, not always reading order. Extraction strategy matters.

In [ ]:
def extract_text_from_pdf(pdf_path: str) -> dict:
    """
    Extract text from each page of a PDF.
    Returns a dict with metadata and a list of page texts.
    """
    doc = fitz.open(pdf_path)

    result = {
        "num_pages": len(doc),
        "pages": []
    }

    for page_num, page in enumerate(doc):
        text = page.get_text("text")  # plain text mode
        result["pages"].append({
            "page": page_num + 1,
            "text": text
        })

    doc.close()
    return result


pdf_data = extract_text_from_pdf(PDF_PATH)
full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])

print(f"Pages : {pdf_data['num_pages']}")
print("\n--- First 500 chars of page 1 (raw) ---")
print(pdf_data['pages'][0]['text'][:500])
print(f"\nTotal characters: {len(full_text_raw):,}")
print(f"Total words: {len(full_text_raw.split()):,}")

Pages : 3

--- First 500 chars of page 1 (raw) ---
Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, kn

Total characters: 4,819
Total words: 404


In [ ]:
import re

def clean_text(text: str) -> str:
    """
    Basic cleaning:
    - Collapse multiple newlines and spaces
    - Remove page numbers and headers (simple heuristic)
    - Strip leading and trailing whitespace
    """
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


full_text_raw = "\n".join(p['text'] for p in pdf_data['pages'])
full_text = clean_text(full_text_raw)

print("--- Cleaned text (first 500 chars) ---")
print(full_text[:500])
print(f"\nTotal characters: {len(full_text):,}")
print(f"Total words: {len(full_text.split()):,}")

--- Cleaned text (first 500 chars) ---
Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&
&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&
&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%
%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%
%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$
$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((
((((((_. is a beautiful country in South Asia, kn

Total characters: 4,792
Total words: 404


In [ ]:
# Count tokens before selecting a qna strategy
enc = tiktoken.get_encoding("cl100k_base")  # #https://tiktokenizer.vercel.app/
total_tokens = len(enc.encode(full_text))

print(f"Total tokens: {total_tokens:,}")
print()
print("Model context limits:")
print(f"  BERT          : 512 tokens  -> fits {512/total_tokens*100:.1f}% of the document")
print(f"  BART-large    : 1024 tokens -> fits {1024/total_tokens*100:.1f}% of the document")
print(f"  GPT-3.5       : 4096 tokens -> fits {4096/total_tokens*100:.1f}% of the document")
print(f"  GPT-4 / Claude: 128K tokens -> fits {min(128000/total_tokens*100, 100):.1f}% of the document")
print()
print("Chunking is required for BART on long documents.")

Total tokens: 1,313

Model context limits:
  BERT          : 512 tokens  -> fits 39.0% of the document
  BART-large    : 1024 tokens -> fits 78.0% of the document
  GPT-3.5       : 4096 tokens -> fits 312.0% of the document
  GPT-4 / Claude: 128K tokens -> fits 100.0% of the document

Chunking is required for BART on long documents.


---
# Block 3: Chunking and QnA

BART have token limits.
For long documents, split text into chunks, embed each chunk, and keep in memory.

Then, for question - embed question, do similarity search based on cosine similarity. then fetch top chunks (< 1024tokens) then send to bart.

bart answers the question.

In [ ]:
def chunk_text(text: str, max_tokens: int = 330, overlap_tokens: int = 15) -> list[str]:
    """
    Split text into token-aware chunks with overlap.

    overlap_tokens: tokens repeated between adjacent chunks
    to preserve context across boundaries.
    """
    enc = tiktoken.get_encoding("cl100k_base")  #https://tiktokenizer.vercel.app/
    tokens = enc.encode(text)

    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_body = enc.decode(chunk_tokens)
        chunks.append(chunk_body)
        start += max_tokens - overlap_tokens

    return chunks


chunks = chunk_text(full_text, max_tokens=330, overlap_tokens=15)

print(f"Total chunks: {len(chunks)}")
print("Target chunk size: 330 tokens")
print("Overlap: 50 tokens")
print()
for i, chunk in enumerate(chunks[:3]):
    tok_count = len(enc.encode(chunk))
    print(f"Chunk {i+1}: {tok_count} tokens | preview: {chunk[:80].strip()!r}")

Total chunks: 5
Target chunk size: 330 tokens
Overlap: 50 tokens

Chunk 1: 330 tokens | preview: 'Bangladesh(_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$$$&&&&&&&&&&&&&&\n&&&&&&&()))))'
Chunk 2: 330 tokens | preview: '$$$$$$\n$$&&&&&&&&&&&&&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%'
Chunk 3: 330 tokens | preview: '&&&&&&&&&())))))))))(((((((((((((((_$$$$$$$$$$$$$$$$$$%%%%%%%%%%%%%%$$$$$$\n$$&&&'


In [ ]:
!pip install --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 64.1 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Using the Qwen3.5-0.8B model as requested
model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True, device_map="auto", torch_dtype="auto")

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Qwen3.5-0.8B model loaded on {device}")

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Qwen3.5-0.8B model loaded on cuda


In [ ]:
from sentence_transformers import SentenceTransformer, util
import re

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks, convert_to_tensor=True)

def ask_question(question: str, top_k: int = 5):
    # 1. Similarity Search
    question_embedding = embedder.encode(question, convert_to_tensor=True)
    hits = util.semantic_search(question_embedding, chunk_embeddings, top_k=top_k)[0]
    context = " ".join([chunks[hit['corpus_id']] for hit in hits])
    # print(context)

    # 2. Construct a ChatML style prompt for Qwen
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Answer the question concisely using only the provided context. If the answer is not in the context, say you do not know."},
        {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # 3. Generate Answer
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    generated_ids = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    # 4. Filter out the prompt and clean thinking tags
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    if "assistant\n" in response:
        answer = response.split("assistant\n")[-1].strip()
    else:
        answer = response[len(prompt):].strip()

    # Remove <think>...</think> blocks and extra whitespace
    answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()

    return answer

print("RAG Pipeline (Clean Output) Ready.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG Pipeline (Clean Output) Ready.


In [ ]:
query = "What is the capital city of Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the capital city of Bangladesh?
Answer: Dhaka.


In [ ]:
query = "Which river is considered the lifeline of Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: Which river is considered the lifeline of Bangladesh?
Answer: The Padma River is considered the lifeline of Bangladesh.


In [ ]:
query = "Which river is considered the lifeline of Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: Which river is considered the lifeline of Bangladesh?
Answer: The Padma River is considered the lifeline of Bangladesh.


In [ ]:
query = "In which year did Bangladesh gain independence?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: In which year did Bangladesh gain independence?
Answer: 1971


In [ ]:
query = "What is the name of the world’s largest mangrove forest located in Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the name of the world’s largest mangrove forest located in Bangladesh?
Answer: The Sundarbans.


In [ ]:
query = "What is the official language of Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: What is the official language of Bangladesh?
Answer: Bengali (also known as Bangla).


In [ ]:
query = "Which currency is used in Bangladesh? What is the national animal of Bangladesh?"
answer = ask_question(query)

print(f"Question: {query}")
print(f"Answer: {answer}")

Question: Which currency is used in Bangladesh? What is the national animal of Bangladesh?
Answer: The official currency of Bangladesh is the **Bangladeshi Taka**. The national animal of Bangladesh is the **Royal Bengal Tiger**.
